In [ ]:
import sys

print (sys.executable)

In [ ]:
import numpy as np
import pandas as pd
import sklearn

print (np.__version__)
print (pd.__version__)
print (sklearn.__version__)

## Analyse exploratoire des données d’attrition — TechNova Partners

Ce notebook a pour objectif de réaliser une analyse exploratoire des trois fichiers fournis dans le cadre de la mission RH chez **TechNova Partners**.

L’enjeu est d’identifier des différences significatives entre les employés ayant quitté l’entreprise et ceux qui y sont encore, afin de préparer la phase de modélisation supervisée.

## Contexte métier

TechNova Partners, une ESN spécialisée dans le conseil en transformation digitale et la vente d’applications SaaS, fait face à un niveau de turnover plus élevé qu’à l’habitude.

Le département RH souhaite comprendre les causes potentielles de départ des employés à partir de plusieurs sources de données internes :

- un extrait du **SIRH** ;
- un extrait des **évaluations de performance** ;
- un extrait du **sondage collaborateurs**.

L’objectif de cette première étape est de structurer, nettoyer et explorer ces données afin de faire émerger des premiers facteurs associés à l’attrition.

## Objectif du notebook

Ce notebook poursuit quatre objectifs principaux :

1. charger et examiner les trois fichiers de départ ;
2. identifier les colonnes et les types de variables ;
3. construire un **DataFrame central** regroupant les informations pertinentes ;
4. produire des statistiques descriptives et des visualisations permettant de comparer les employés ayant quitté l’entreprise à ceux qui y travaillent encore.

## Prérequis

Avant de commencer l’analyse, les éléments suivants ont été préparés :

- un environnement virtuel Python contenant les packages nécessaires au projet ;
- un notebook dédié à l’analyse exploratoire ;
- une structure de projet organisée avec les fichiers de données placés dans le dossier `data/raw/`.

## 1. Imports et configuration de l’environnement

Cette section charge les bibliothèques nécessaires à l’analyse :

- `pandas` et `numpy` pour la manipulation de données ;
- `matplotlib` et `seaborn` pour les visualisations ;
- `missingno` pour l’analyse visuelle des valeurs manquantes.

Des options d’affichage sont également définies afin de faciliter la lecture des DataFrames.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

pd.set_option ("display.max_columns", 200)
pd.set_option ("display.max_rows", 200)

sns.set_theme (style="whitegrid")

In [ ]:
PROJECT_ROOT = Path.cwd ( ).resolve ( ).parent if Path.cwd ( ).name == "notebooks" else Path.cwd ( ).resolve ( )

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

DATA_RAW.mkdir (parents=True, exist_ok=True)
DATA_PROCESSED.mkdir (parents=True, exist_ok=True)

print ("PROJECT_ROOT :", PROJECT_ROOT)
print ("DATA_RAW :", DATA_RAW)
print ("DATA_PROCESSED :", DATA_PROCESSED)

In [ ]:
list (DATA_RAW.glob ("*"))

In [ ]:
csv_files = list (DATA_RAW.glob ("*.csv"))

if not csv_files:
    print ("Aucun fichier CSV trouvé dans :", DATA_RAW)
else:
    file_path = csv_files[0]
    print ("Fichier trouvé :", file_path)

    df_central = pd.read_csv (file_path, encoding="utf-8-sig")

    print ("Shape :", df_central.shape)
    display (df_central.head ( ))

In [ ]:
output_path = DATA_PROCESSED / "df_central.csv"

df_central.to_csv (
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print ("Fichier sauvegardé :", output_path)
print ("Shape :", df_central.shape)

In [ ]:
print (df_central.columns.tolist ( ))

In [ ]:
csv_files = list (DATA_RAW.glob ("*.csv"))

if not csv_files:
    print ("Aucun fichier CSV trouvé dans :", DATA_RAW)
else:
    file_path = csv_files[0]
    print ("Fichier trouvé :", file_path)

    df_central = pd.read_csv (file_path, encoding="utf-8-sig")

    print ("Shape :", df_central.shape)
    display (df_central.head ( ))

In [ ]:
for f in DATA_RAW.iterdir ( ):
    print (f.name)

In [ ]:
csv_files = list (DATA_RAW.glob ("*.csv"))

if not csv_files:
    raise FileNotFoundError (f"Aucun fichier CSV trouvé dans {DATA_RAW}")

file_path = csv_files[0]
print ("Fichier source chargé :", file_path)

df_central = pd.read_csv (file_path, encoding="utf-8-sig")

print ("Shape :", df_central.shape)
display (df_central.head ( ))

In [ ]:
output_path = DATA_PROCESSED / "df_central.csv"

df_central.to_csv (
    output_path,
    index=False,
    encoding="utf-8-sig"
)

print ("Fichier sauvegardé :", output_path)
print ("Shape :", df_central.shape)

In [ ]:
type (df_central), df_central.shape

In [ ]:
df_central = pd.read_csv (DATA_PROCESSED / "df_central.csv", encoding="utf-8-sig")
df_central.head ( )

## 2. Définition des chemins du projet

Afin de rendre le notebook réutilisable, les chemins vers le dossier racine du projet et vers le répertoire `data/raw/` sont définis dynamiquement.

Cette approche permet d’éviter les chemins absolus spécifiques à une machine et facilite l’exécution du notebook dans différents environnements.

In [ ]:
PROJECT_ROOT = Path.cwd ( ).resolve ( ).parent if Path.cwd ( ).name == "notebooks" else Path.cwd ( ).resolve ( )
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print ("PROJECT_ROOT :", PROJECT_ROOT)
print ("DATA_RAW     :", DATA_RAW)

## 3. Chargement des fichiers source

Les trois fichiers fournis sont maintenant chargés :

- le fichier **SIRH** ;
- le fichier des **évaluations** ;
- le fichier du **sondage**.

Cette étape constitue la base de toute l’analyse exploratoire. Avant toute transformation, il est essentiel de vérifier que les fichiers sont bien lisibles et que leur structure correspond à ce qui est attendu.

In [ ]:
sirh_path = DATA_RAW / "extrait_sirh.csv"
eval_path = DATA_RAW / "extrait_eval.csv"
survey_path = DATA_RAW / "extrait_sondage.csv"

df_sirh = pd.read_csv (sirh_path)
df_eval = pd.read_csv (eval_path)
df_survey = pd.read_csv (survey_path)

## 4. Première inspection des jeux de données

Chaque fichier est inspecté selon plusieurs dimensions :

- nombre de lignes et de colonnes ;
- noms des colonnes ;
- types de variables ;
- volume de valeurs manquantes ;
- présence éventuelle de doublons ;
- aperçu des premières lignes.

Cette première lecture permet de mieux comprendre la structure des données avant de commencer leur préparation.

In [ ]:
def inspect_df(df: pd.DataFrame, name: str) -> None:
    print ("=" * 80)
    print (name)
    print ("=" * 80)
    print ("Shape :", df.shape)
    print ("\nColonnes :")
    print (df.columns.tolist ( ))
    print ("\nTypes :")
    print (df.dtypes)
    print ("\nValeurs manquantes :")
    print (df.isna ( ).sum ( ).sort_values (ascending=False).head (20))
    print ("\nDoublons :", df.duplicated ( ).sum ( ))
    display (df.head ( ))

In [ ]:
inspect_df (df_sirh, "SIRH")
inspect_df (df_eval, "ÉVALUATIONS")
inspect_df (df_survey, "SONDAGE")

In [ ]:
def summarize_missing(df: pd.DataFrame, name: str) -> None:
    missing_count = df.isna ( ).sum ( ).sum ( )
    missing_pct = (missing_count / df.size) * 100

    print (f"{name}")
    print (f"- valeurs manquantes totales : {missing_count}")
    print (f"- pourcentage de valeurs manquantes : {missing_pct:.2f}%")
    print ( )


summarize_missing (df_sirh, "SIRH")
summarize_missing (df_eval, "Évaluations")
summarize_missing (df_survey, "Sondage")

## 5. Vérification des valeurs manquantes

Une vérification des valeurs manquantes a été réalisée sur les trois jeux de données.

Résultat :
- aucune valeur manquante n’a été détectée dans le fichier SIRH ;
- aucune valeur manquante n’a été détectée dans le fichier des évaluations ;
- aucune valeur manquante n’a été détectée dans le fichier du sondage.

Cette étape confirme que les trois extraits sont complets sur le plan des valeurs manquantes, ce qui simplifie la phase de préparation des données.

In [ ]:
def check_empty_strings(df: pd.DataFrame, name: str) -> None:
    empty_count = (
        df.astype (str)
        .apply (lambda col: col.str.strip ( ) == "")
        .sum ( )
        .sum ( )
    )
    print (f"{name} - chaînes vides : {empty_count}")


check_empty_strings (df_sirh, "SIRH")
check_empty_strings (df_eval, "Évaluations")
check_empty_strings (df_survey, "Sondage")

## 6. Normalisation des noms de colonnes

Les noms de colonnes sont harmonisés afin de faciliter les manipulations ultérieures :

- passage en minuscules ;
- suppression des espaces ;
- remplacement des caractères accentués ;
- suppression des caractères spéciaux.

Cette normalisation permet de travailler avec des noms de colonnes cohérents, plus simples à appeler dans le code et moins sujets aux erreurs.

In [ ]:
def clean_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy ( )
    df.columns = (
        df.columns
        .str.strip ( )
        .str.lower ( )
        .str.replace (" ", "_", regex=False)
        .str.replace ("é", "e", regex=False)
        .str.replace ("è", "e", regex=False)
        .str.replace ("ê", "e", regex=False)
        .str.replace ("à", "a", regex=False)
        .str.replace ("ç", "c", regex=False)
        .str.replace (r"[^a-z0-9_]", "", regex=True)
    )
    return df


df_sirh = clean_columns (df_sirh)
df_eval = clean_columns (df_eval)
df_survey = clean_columns (df_survey)

print (df_sirh.columns.tolist ( ))
print (df_eval.columns.tolist ( ))
print (df_survey.columns.tolist ( ))

## 7. Recherche d’une clé de rapprochement entre les fichiers

Une étape importante de l’analyse consiste à identifier les colonnes permettant de rapprocher les trois sources de données.

L’objectif ici est de vérifier s’il existe :

- une clé commune explicite entre les fichiers ;
- ou, à défaut, une structure de données permettant un rapprochement cohérent.

In [ ]:
print ("SIRH")
print (df_sirh.columns.tolist ( ))

print ("\nEVAL")
print (df_eval.columns.tolist ( ))

print ("\nSURVEY")
print (df_survey.columns.tolist ( ))

In [ ]:
common_sirh_eval = set (df_sirh.columns).intersection (df_eval.columns)
common_sirh_survey = set (df_sirh.columns).intersection (df_survey.columns)
common_eval_survey = set (df_eval.columns).intersection (df_survey.columns)

print ("Colonnes communes SIRH / EVAL :", common_sirh_eval)
print ("Colonnes communes SIRH / SURVEY :", common_sirh_survey)
print ("Colonnes communes EVAL / SURVEY :", common_eval_survey)

In [ ]:
for name, df in {
    "SIRH": df_sirh,
    "EVAL": df_eval,
    "SURVEY": df_survey
}.items ( ):
    print (f"\n{name}")
    for col in df.columns:
        if "id" in col.lower ( ) or "matric" in col.lower ( ) or "employ" in col.lower ( ) or "collab" in col.lower ( ):
            print (col)

In [ ]:
print ("Colonnes communes aux 3 fichiers :")
common_all = set (df_sirh.columns).intersection (df_eval.columns).intersection (df_survey.columns)
print (common_all)

print ("\nColonnes qui ressemblent à une clé :")
for col in sorted (common_all):
    if any (token in col.lower ( ) for token in ["id", "matric", "employ", "collab"]):
        print (col)

In [ ]:
common_all = set (df_sirh.columns).intersection (df_eval.columns).intersection (df_survey.columns)

candidate_keys = [
    col for col in sorted (common_all)
    if any (token in col.lower ( ) for token in ["id", "matric", "employ", "collab"])
]

print ("Colonnes communes aux 3 fichiers :", common_all)
print ("Candidates de clé :", candidate_keys)

if len (candidate_keys) == 1:
    key = candidate_keys[0]
    print (f"\nClé probable retenue : {key}")
    print (df_sirh[key].nunique ( ), df_sirh.shape[0])
    print (df_eval[key].nunique ( ), df_eval.shape[0])
    print (df_survey[key].nunique ( ), df_survey.shape[0])
else:
    print ("\nAucune clé unique évidente trouvée automatiquement.")
    print ("Vérifiez manuellement les colonnes affichées ci-dessus.")

In [ ]:
for name, df in {
    "SIRH": df_sirh,
    "EVAL": df_eval,
    "SURVEY": df_survey
}.items ( ):
    print (f"\n{'=' * 80}\n{name}\n{'=' * 80}")
    for col in df.columns:
        if any (token in col.lower ( ) for token in [
            "id", "matric", "employ", "collab", "salar", "person", "agent"
        ]):
            print (col)

In [ ]:
for name, df in {
    "SIRH": df_sirh,
    "EVAL": df_eval,
    "SURVEY": df_survey
}.items ( ):
    print (f"\n{'=' * 80}\n{name}\n{'=' * 80}")
    for col in df.columns:
        print (col)

In [ ]:
sirh_cols = set (df_sirh.columns)
eval_cols = set (df_eval.columns)
survey_cols = set (df_survey.columns)

print ("df_sirh columns:")
print (df_sirh.columns.tolist ( ))

print ("\ndf_eval columns:")
print (df_eval.columns.tolist ( ))

print ("\ndf_survey columns:")
print (df_survey.columns.tolist ( ))

In [ ]:
print ("Shapes")
print ("SIRH  :", df_sirh.shape)
print ("EVAL  :", df_eval.shape)
print ("SURVEY:", df_survey.shape)

In [ ]:
from pathlib import Path
import pandas as pd


def clean_columns(df):
    df = df.copy ( )
    df.columns = (
        pd.Index (df.columns)
        .astype (str)
        .str.strip ( )
        .str.lower ( )
        .str.replace ("\ufeff", "", regex=False)
        .str.replace (" ", "_", regex=False)
        .str.replace ("-", "_", regex=False)
    )
    return df


def rename_id_column(df, df_name="df"):
    df = clean_columns (df)

    possible_id_names = [
        "id_employee",
        "employee_id",
        "id_employe",
        "id_collaborateur",
        "collaborateur_id",
        "matricule",
        "id",
        "unnamed:_0",
        "unnamed:_0.1",
    ]

    for col in possible_id_names:
        if col in df.columns:
            df = df.rename (columns={col: "id_employee"})
            print (f"{df_name} -> colonne clé trouvée : {col} -> id_employee")
            return df

    print (f"{df_name} -> aucune colonne clé trouvée automatiquement")
    print ("Colonnes disponibles :", df.columns.tolist ( ))
    return df


df_sirh = rename_id_column (df_sirh, "df_sirh")
df_eval = rename_id_column (df_eval, "df_eval")
df_survey = rename_id_column (df_survey, "df_survey")

In [ ]:
# 1) Vérifier les noms de colonnes réels
print ("df_sirh :", df_sirh.columns.tolist ( ))
print ("df_eval :", df_eval.columns.tolist ( ))
print ("df_survey :", df_survey.columns.tolist ( ))

In [ ]:
def normalize_columns(df):
    df = df.copy ( )
    df.columns = (
        df.columns.astype (str)
        .str.strip ( )
        .str.lower ( )
        .str.replace ("\ufeff", "", regex=False)
        .str.replace (" ", "_", regex=False)
        .str.replace ("-", "_", regex=False)
    )
    return df


df_sirh = normalize_columns (df_sirh)
df_eval = normalize_columns (df_eval)
df_survey = normalize_columns (df_survey)

print ("df_sirh :", df_sirh.columns.tolist ( ))
print ("df_eval :", df_eval.columns.tolist ( ))
print ("df_survey :", df_survey.columns.tolist ( ))

In [ ]:
def force_id_employee(df, name):
    candidates = [
        "id_employee",
        "employee_id",
        "id_employe",
        "id_collaborateur",
        "collaborateur_id",
        "matricule",
        "id"
    ]

    for col in candidates:
        if col in df.columns:
            df = df.rename (columns={col: "id_employee"})
            print (f"{name} -> {col} renommée en id_employee")
            return df

    print (f"{name} -> aucune colonne ID trouvée automatiquement")
    print (df.columns.tolist ( ))
    return df


df_sirh = force_id_employee (df_sirh, "df_sirh")
df_eval = force_id_employee (df_eval, "df_eval")
df_survey = force_id_employee (df_survey, "df_survey")

In [ ]:
print ("id_employee dans df_sirh ?", "id_employee" in df_sirh.columns)
print ("id_employee dans df_eval ?", "id_employee" in df_eval.columns)
print ("id_employee dans df_survey ?", "id_employee" in df_survey.columns)

In [ ]:
for name, df in {
    "df_sirh": df_sirh,
    "df_eval": df_eval,
    "df_survey": df_survey,
}.items ( ):
    print (name, "->", "id_employee" in df.columns)

In [ ]:
print (df_eval.head ( ))
print (df_survey.head ( ))

In [ ]:
df_eval = df_eval.rename (columns={"employee_id": "id_employee"})
df_survey = df_survey.rename (columns={"id_employe": "id_employee"})

In [ ]:
df_sirh = df_sirh.reset_index (drop=True)
df_eval = df_eval.reset_index (drop=True)
df_survey = df_survey.reset_index (drop=True)

df_central = pd.concat ([df_sirh, df_eval, df_survey], axis=1)

print ("Shape df_central :", df_central.shape)
display (df_central.head ( ))

In [ ]:
print ("df_sirh :", df_sirh.columns.tolist ( ))
print ("df_eval :", df_eval.columns.tolist ( ))
print ("df_survey :", df_survey.columns.tolist ( ))

In [ ]:
print ("Colonnes dupliquées :", df_central.columns[df_central.columns.duplicated ( )].tolist ( ))
print ("Répartition de la cible :")
print (df_central["a_quitte_l_entreprise"].value_counts (dropna=False))
print (df_central["a_quitte_l_entreprise"].value_counts (normalize=True, dropna=False))

In [ ]:
print ("Doublons id_employee dans SIRH :", df_sirh["id_employee"].duplicated ( ).sum ( ))
print ("Valeurs manquantes id_employee dans SIRH :", df_sirh["id_employee"].isna ( ).sum ( ))

df_sirh = df_sirh.reset_index (drop=True)
df_eval = df_eval.reset_index (drop=True)
df_survey = df_survey.reset_index (drop=True)

df_central = pd.concat ([df_sirh, df_eval, df_survey], axis=1)

print ("Shape df_central :", df_central.shape)
display (df_central.head ( ))

## 8. Choix de la méthode de rapprochement

Les trois fichiers ne partagent pas de clé de jointure explicite commune. En revanche, ils présentent le même nombre de lignes (1470), ce qui suggère un alignement ligne à ligne entre les sources.

Le fichier **SIRH** contient une clé explicite, `id_employee`, mais cette clé n’apparaît pas dans les fichiers **Évaluations** et **Sondage**. Dans ce contexte, le DataFrame central a donc été construit par **concaténation horizontale après réinitialisation des index**.

Cette décision repose sur l’hypothèse que les trois extraits sont ordonnés ligne à ligne selon les mêmes employés, ce qui est cohérent avec la structure pédagogique du jeu de données.

## 9. Construction et sécurisation de la variable cible

La variable cible du projet est `a_quitte_l_entreprise`.

Elle indique si un employé a quitté l’entreprise ou non :

- `Oui` : employé ayant quitté l’entreprise ;
- `Non` : employé encore présent.

La colonne est ici standardisée afin de fiabiliser les analyses descriptives et les futures étapes de modélisation.

In [ ]:
df_central["a_quitte_l_entreprise"] = (
    df_central["a_quitte_l_entreprise"]
    .astype (str)
    .str.strip ( )
    .str.capitalize ( )
)

print (df_central["a_quitte_l_entreprise"].value_counts (dropna=False))

## 10. Encodage binaire de la cible

Une version binaire de la variable cible est créée afin de faciliter certaines analyses quantitatives et la future modélisation :

- `1` pour les employés ayant quitté l’entreprise ;
- `0` pour les employés restés dans l’entreprise.

In [ ]:
df_central["attrition_bin"] = df_central["a_quitte_l_entreprise"].apply (
    lambda x: 1 if x == "Oui" else 0
)

print (df_central["attrition_bin"].value_counts ( ))

## 11. Typologie des variables

Les variables sont séparées en deux groupes :

- les variables **numériques**, adaptées aux statistiques descriptives, corrélations et boxplots ;
- les variables **catégorielles**, adaptées aux tableaux croisés et graphiques de répartition.

Cette distinction est essentielle pour choisir les bons outils d’exploration en fonction de la nature des données.

In [ ]:
print (df_central.dtypes.sort_index ( ))

In [ ]:
num_cols = df_central.select_dtypes (include=["number"]).columns.tolist ( )
cat_cols = df_central.select_dtypes (include=["object", "string", "category"]).columns.tolist ( )

print ("Colonnes numériques :")
print (num_cols)

print ("\nColonnes catégorielles :")
print (cat_cols)

## 12. Statistiques descriptives globales

Des statistiques descriptives sont calculées sur chacun des trois fichiers de départ ainsi que sur le DataFrame central consolidé.

Cette étape permet d’obtenir une première vue d’ensemble des distributions, des moyennes, des extrêmes et des modalités présentes dans les données.

In [ ]:
display (df_sirh.describe (include="all").T)
display (df_eval.describe (include="all").T)
display (df_survey.describe (include="all").T)
display (df_central.describe (include="all").T)

In [ ]:
display (df_central[num_cols].describe ( ).T)

## 13. Comparaison des variables selon l’attrition

Les variables numériques sont maintenant comparées entre deux groupes :

- les employés ayant quitté l’entreprise ;
- les employés restés dans l’entreprise.

L’objectif est de faire émerger les premières différences de profil entre ces deux populations.

In [ ]:
mean_by_attrition = df_central.groupby ("a_quitte_l_entreprise")[num_cols].mean ( ).T
median_by_attrition = df_central.groupby ("a_quitte_l_entreprise")[num_cols].median ( ).T

display (mean_by_attrition)
display (median_by_attrition)

In [ ]:
comparison = pd.DataFrame ({
    "moyenne_non": df_central[df_central["a_quitte_l_entreprise"] == "Non"][num_cols].mean ( ),
    "moyenne_oui": df_central[df_central["a_quitte_l_entreprise"] == "Oui"][num_cols].mean ( ),
})
comparison["ecart_oui_moins_non"] = comparison["moyenne_oui"] - comparison["moyenne_non"]

display (comparison.sort_values ("ecart_oui_moins_non", ascending=False))

## 14. Analyse des variables catégorielles

Les variables qualitatives sont étudiées à l’aide de tableaux croisés afin d’observer la répartition des départs selon différentes catégories, par exemple le genre, le département, le poste ou encore le niveau d’éducation.

Les tableaux normalisés permettent d’identifier les catégories présentant un taux d’attrition relativement élevé.

In [ ]:
for col in ["genre", "statut_marital", "departement", "poste", "niveau_education", "frequence_deplacement"]:
    if col in df_central.columns:
        print (f"\n===== {col} =====")
        display (pd.crosstab (df_central[col], df_central["a_quitte_l_entreprise"], margins=True))
        display (pd.crosstab (df_central[col], df_central["a_quitte_l_entreprise"], normalize="index"))

## 15. Visualisations exploratoires

Des graphiques sont générés afin de rendre les résultats plus lisibles et plus parlants.

Le choix des visualisations est adapté à la nature des variables :

- `countplot` pour la répartition de la cible ;
- `boxplot` pour comparer une variable quantitative selon l’attrition ;
- graphiques en barres sur taux normalisés pour les variables catégorielles ;
- heatmap pour les corrélations entre variables numériques.

In [ ]:
sns.countplot (data=df_central, x="a_quitte_l_entreprise")
plt.title ("Répartition des employés ayant quitté l'entreprise")
plt.show ( )

In [ ]:
sns.boxplot (data=df_central, x="a_quitte_l_entreprise", y="age")
plt.title ("Âge selon l'attrition")
plt.show ( )

In [ ]:
sns.boxplot (data=df_central, x="a_quitte_l_entreprise", y="annees_dans_l_entreprise")
plt.title ("Ancienneté dans l'entreprise selon l'attrition")
plt.show ( )

In [ ]:
sns.boxplot (data=df_central, x="a_quitte_l_entreprise", y="nombre_heures_travailless")
plt.title ("Nombre d'heures travaillées selon l'attrition")
plt.show ( )

In [ ]:
if "heure_supplementaires" in df_central.columns:
    sns.countplot (data=df_central, x="heure_supplementaires", hue="a_quitte_l_entreprise")
    plt.title ("Heures supplémentaires et attrition")
    plt.show ( )

In [ ]:
satisfaction_cols = [
    "satisfaction_employee_environnement",
    "satisfaction_employee_nature_travail",
    "satisfaction_employee_equipe",
    "satisfaction_employee_equilibre_pro_perso",
]

for col in satisfaction_cols:
    if col in df_central.columns:
        sns.boxplot (data=df_central, x="a_quitte_l_entreprise", y=col)
        plt.title (f"{col} selon l'attrition")
        plt.show ( )

In [ ]:
attrition_by_dep = pd.crosstab (
    df_central["departement"],
    df_central["a_quitte_l_entreprise"],
    normalize="index"
)

if "Oui" in attrition_by_dep.columns:
    attrition_by_dep["Oui"].sort_values (ascending=False).plot (kind="bar", figsize=(10, 5))
    plt.title ("Taux d'attrition par département")
    plt.ylabel ("Proportion de départs")
    plt.show ( )

In [ ]:
attrition_by_poste = pd.crosstab (
    df_central["poste"],
    df_central["a_quitte_l_entreprise"],
    normalize="index"
)

if "Oui" in attrition_by_poste.columns:
    attrition_by_poste["Oui"].sort_values (ascending=False).plot (kind="bar", figsize=(12, 5))
    plt.title ("Taux d'attrition par poste")
    plt.ylabel ("Proportion de départs")
    plt.show ( )

## 16. Corrélations entre variables numériques

Une matrice de corrélation est calculée pour mieux comprendre les relations linéaires entre les variables numériques.

Une attention particulière est portée à la corrélation avec la variable cible binaire `attrition_bin`, ce qui permet de repérer les variables potentiellement les plus liées au départ d’un employé.

In [ ]:
import numpy as np

plt.figure (figsize=(14, 10))
corr = df_central[num_cols].corr (numeric_only=True)
mask = np.triu (np.ones_like (corr, dtype=bool))

sns.heatmap (
    corr,
    mask=mask,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".2f",
    square=True
)

plt.title ("Matrice de corrélation des variables numériques")
plt.show ( )

## 17. Synthèse intermédiaire

À ce stade de l’analyse exploratoire :

- les trois fichiers ont été chargés et inspectés ;
- un DataFrame central a été construit ;
- la variable cible a été identifiée et encodée ;
- des statistiques descriptives et des visualisations ont été produites.

Les premiers résultats suggèrent que certaines variables, comme l’ancienneté, le revenu mensuel, le niveau hiérarchique, l’expérience totale ou certains indicateurs de satisfaction, semblent associées à l’attrition.

Ces constats restent exploratoires et devront être confirmés dans la phase de modélisation supervisée.

In [ ]:
cols_for_corr = [col for col in num_cols if col != "attrition_bin"] + ["attrition_bin"]

corr_target = (
    df_central[cols_for_corr]
    .corr (numeric_only=True)["attrition_bin"]
    .sort_values (ascending=False)
)

display (corr_target)

In [ ]:
display (corr_target.drop ("attrition_bin"))

~~~~~~~~## Conclusion

Cette première étape a permis de structurer les données RH issues de plusieurs sources et de mettre en évidence des différences entre les employés ayant quitté l’entreprise et ceux qui y sont encore.

L’analyse exploratoire constitue une base solide pour :

- la préparation des variables ;
- la construction d’un modèle de classification ;
- l’interprétation future des facteurs explicatifs de l’attrition.

La prochaine étape consistera à préparer les données pour la modélisation, entraîner plusieurs algorithmes de classification et comparer leurs performances.

### Lecture des corrélations avec la cible

Les corrélations observées avec `attrition_bin` restent globalement modérées, ce qui est cohérent dans un problème RH de ce type.

Plusieurs variables présentent une corrélation négative avec l’attrition, notamment :

- l’expérience totale ;
- le niveau hiérarchique du poste ;
- le revenu mensuel ;
- l’âge ;
- l’ancienneté dans l’entreprise et dans le poste actuel.

Cela suggère que les employés les plus expérimentés, les mieux rémunérés ou les plus installés dans l’organisation semblent moins susceptibles de quitter l’entreprise.

À l’inverse, `distance_domicile_travail` présente une corrélation positive, certes faible, mais allant dans le sens d’un risque de départ légèrement plus élevé lorsque la distance domicile-travail augmente.

Ces résultats doivent rester interprétés avec prudence : ils mettent en évidence des associations statistiques, mais ne constituent pas à eux seuls des relations causales.

In [ ]:
PROJECT_ROOT = Path.cwd ( ).resolve ( ).parent if Path.cwd ( ).name == "notebooks" else Path.cwd ( ).resolve ( )
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir (parents=True, exist_ok=True)

df_central.to_csv (DATA_PROCESSED / "df_central.csv", index=False)

print ("Fichier sauvegardé :", DATA_PROCESSED / "df_central.csv")

## Sauvegarde du DataFrame central

Le DataFrame central final est sauvegardé dans le dossier `data/processed` afin d’être réutilisé dans les étapes suivantes du projet, notamment pour la préparation des données et la modélisation.